## Prepare

In [1]:
%cd /Users/juntao/src/github.com/lance/lance/python

/Users/juntao/src/github.com/lance/lance/python


macos intel chip
```bash
cd python
make install
source .venv/bin/activate
python -m ensurepip --upgrade
pip install --upgrade pip
pip install pandas numpy pyarrow jupyter
```

mac chip
> pip install pylance pandas numpy pyarrow jupyter

In [2]:
import lance; 
import pandas as pd
import os
import shutil
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import time
import pandas as pd
from itertools import islice

def delete_if_exists(p):
    if os.path.exists(p):
        if os.path.isfile(p):
            os.remove(p)
            print(f"✅ 成功删除文件: {p}")
        if os.path.isdir(p):
            shutil.rmtree(p)
            print(f"✅ 成功删除目录: {p}")

def print_lance(p):
    print("=== 数据集结构 ===")
    for root, dirs, files in os.walk(p):
        indent = " " * 2
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 2 * 2
        for file in files:
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath)
            print(f"{subindent}{file} ({size:,} bytes)")

In [ ]:
print(lance.__version__)

## File Version 
- 用 `xxd` 或十六进制编辑器查看 Lance 文件尾部，找到 "LANC"
- 000003004c414e43 -> 0(小端) 3(小端) L A N C
   - `try_from_major_minor`: 0 3 | 2 0 => 2.0

In [ ]:
delete_if_exists("/tmp/day1.lance")

df = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "score": [85.5, 92.0, 78.5]
})

# ds = lance.write_dataset(df, "/tmp/day1.lance", data_storage_version="2.2")
ds = lance.write_dataset(df, "/tmp/day1.lance")
print(f"Created dataset with {ds.count_rows()} rows")

# 4. 读取数据
ds = lance.dataset("/tmp/day1.lance")
print("格式版本:", ds.data_storage_version)
print(ds.to_table().to_pandas())

In [ ]:
fragment = ds.get_fragment(0)
fragment.data_files()[0]

## Lance vs Parquet

In [218]:
import lance
import pyarrow as pa
import pyarrow.parquet as pq
import time
import numpy as np
import pandas as pd


delete_if_exists("/tmp/day2.parquet")

# 1. 创建测试数据（100万行）
n = 1_000_000
df = pd.DataFrame({
    "id": range(n),
    "value": np.random.randn(n),
    "category": np.random.choice(["A", "B", "C"], n)
})

# 2. 写入 Parquet
start = time.time()
pq.write_table(pa.Table.from_pandas(df), "/tmp/day2.parquet")
parquet_write_time = time.time() - start
print(f"Parquet write: {parquet_write_time:.2f}s")

# 3. 写入 Lance
start = time.time()
lance.write_dataset(df, "/tmp/day2.lance", mode="overwrite")
lance_write_time = time.time() - start
print(f"Lance write: {lance_write_time:.2f}s")

# 模拟随机访问：读取100个随机行
random_indices = np.random.choice(n, 100, replace=False)

# 4. 随机访问对比 - Parquet
start = time.time()
parquet_table = pq.read_table("/tmp/day2.parquet", columns=["value"])
parquet_values = parquet_table.column("value").to_pylist()
parquet_random = [parquet_values[i] for i in random_indices]
parquet_random_time = time.time() - start
print(f"Parquet random access: {parquet_random_time*1000:.1f}ms")

# 5. 随机访问对比 - Lance
start = time.time()
ds = lance.dataset("/tmp/day2.lance")
lance_random = ds.take(random_indices.tolist(), columns=["value"])
lance_random_time = time.time() - start
print(f"Lance random access: {lance_random_time*1000:.1f}ms")

# 6. 对比结果
print(f"\n=== 对比结果 ===")
print(f"随机访问速度提升: {parquet_random_time/lance_random_time:.1f}x")

✅ 成功删除文件: /tmp/day2.parquet
Parquet write: 0.12s
Lance write: 0.24s
Parquet random access: 508.1ms
Lance random access: 18.4ms

=== 对比结果 ===
随机访问速度提升: 27.6x


### 随机访问性能
读取第 N 行的某个列

In [219]:
print(f"Pandas: {df["value"].iloc[1000]}")

start = time.time()
value_1000 = ds.take([1000], columns=["value"])
print("Lance:",value_1000["value"].to_pylist()[0])
print(f"Lance random access: {lance_random_time*1000:.1f}ms")

start = time.time()
value_1000 = parquet_table.slice(offset=1000, length=1)
print("parquet",value_1000["value"].to_pylist()[0])
print(f"Parquet random access: {parquet_random_time*1000:.1f}ms")

Pandas: -0.8959520439005898
Lance: -0.8959520439005898
Lance random access: 18.4ms
parquet -0.8959520439005898
Parquet random access: 508.1ms


In [220]:
ds = lance.dataset("/tmp/day2.lance")
fragment = ds.get_fragment(0)
file = fragment.data_files()[0]
print(ds.data_storage_version)
file

2.0


DataFile(path='011111011110100001100110e49c3f4243b8fe77a67501dc65.lance', fields=[0, 1, 2], column_indices=[0, 1, 2], file_major_version=2, file_minor_version=0, file_size_bytes=25001286)

## Lance Laylout

```
Lance File Layout:
├──────────────────────────────────┤ ◄── 文件开头
│ Data Pages                       │
│   Data Buffer 0*                 │
│   Data Buffer 1*                 │
│   ...                            │
│   Data Buffer BN*                │
├──────────────────────────────────┤
│ Column Metadatas                 │
│ |A| Column 0 Metadata*           │
│     Column 1 Metadata*           │
│     ...                          │
│     Column CN Metadata*          │
├──────────────────────────────────┤
│ Column Metadata Offset Table     │
│ |B| Column 0 Metadata Position*  │
│     Column 0 Metadata Size       │
│     ...                          │
│     Column CN Metadata Position  │
│     Column CN Metadata Size      │
├──────────────────────────────────┤
│ Global Buffers Offset Table      │
│ |C| Global Buffer 0 Position*    │
│     Global Buffer 0 Size         │
│     ...                          │
│     Global Buffer GN Position    │
│     Global Buffer GN Size        │
├──────────────────────────────────┤
│ Footer                           │
│   u64: Offset to column meta 0   │ ← A
│   u64: Offset to CMO table       │ ← B
│   u64: Offset to GBO table       │ ← C
│   u32: Number of global bufs     │
│   u32: Number of columns         │
│   u16: Major version             │
│   u16: Minor version             │
│   "LANC"                         │ ← Magic Number
├──────────────────────────────────┤ ◄── 文件结尾
```



| 字段                     | 大小    | 说明                 |
| ------------------------ | ------- | -------------------- |
| Magic Number             | 4 bytes | "LANC"               |
| Major Version            | 2 bytes | 主版本号             |
| Minor Version            | 2 bytes | 次版本号             |
| Number of columns        | 4 bytes | 列数                 |
| Number of global buffers | 4 bytes | 全局缓冲区数         |
| Offset to CMO table      | 8 bytes | 列元数据偏移表位置   |
| Offset to GBO table      | 8 bytes | 全局缓冲区偏移表位置 |
| Offset to column meta    | 8 bytes | 第一个列元数据位置   |



In [14]:
delete_if_exists("/tmp/day3.lance")
uri = "/tmp/day3.lance"
n = 10000
df = pd.DataFrame({
    "id": range(n),
    "value": np.random.randn(n),
    "value2": np.random.randn(n),
    "category": np.random.choice(["A", "B", "C", "D"], n)
})

start = time.time()
lance.write_dataset(df, uri) # mode="overwrite"
lance_write_time = time.time() - start
print(f"Lance write: {lance_write_time:.2f}s")

✅ 成功删除目录: /tmp/day3.lance
Lance write: 0.01s


In [15]:
# 1. Append（追加新数据）
new_df = pd.DataFrame({
  "id": [10001, 10002],
  "value": [1.0, 2.0],
  "value2": [3.0, 4.0],
  "category": ["A", "B"]
})
lance.write_dataset(new_df, uri, mode="append")

In [16]:
ds = lance.dataset(uri)
# ds.delete("id = 10001")           # 删除 id=10001 的行
ds.delete("category = 'A'")

{'num_deleted_rows': 2521}

In [17]:
new_df = pd.DataFrame({
  "id": [100013],
  "value": [1.0],
  "value2": [6.0],
  "category": ["C"]
})
lance.write_dataset(new_df, uri, mode="append")

In [18]:
print_lance(uri)

=== 数据集结构 ===
  day3.lance/
  _deletions/
  1-2-13001175789353526098.arrow (698 bytes)
  0-2-5270388503659910328.arrow (6,586 bytes)
  _versions/
  18446744073709551611.manifest (1,480 bytes)
  latest_version_hint.json (13 bytes)
  18446744073709551612.manifest (1,533 bytes)
  18446744073709551613.manifest (1,357 bytes)
  18446744073709551614.manifest (1,407 bytes)
  _transactions/
  2-44fa6620-eb7f-4bb1-bb1b-8c8869223e30.txn (264 bytes)
  3-30891260-2255-4bdd-a85b-6370950547f5.txn (126 bytes)
  0-221500f6-292d-407f-bcfb-9f557da0ce62.txn (261 bytes)
  1-bc527539-d5a8-42e8-9faa-d18ff2014064.txn (126 bytes)
  data/
  00100011110010111010110095f5e84ffd8857b04fe953f04f.lance (2,017 bytes)
  110110001100000000001011fba35a4218b770dcb09e66d77b.lance (2,017 bytes)
  0101011101011001111110013064df40cb8bccb8b30451c0e0.lance (228,661 bytes)


### 查看 Fragment 结构
- _versions 相当于 manifests
- manifest 包含完整的 fragments
- 比如：version3 包含2个 fragments
   - Fragment 中 deletion_file 是option，如果有删除与lance file 是一一对应的

In [19]:
import lance
import os
import pandas as pd
p = "/tmp/day3.2.lance"
delete_if_exists(p)

# 1. 创建初始数据
df1 = pd.DataFrame({
    "id": [1, 2, 3],
    "version": ["v1", "v1", "v1"],
    "type":[1, 2, 1],
})
ds = lance.write_dataset(df1, p)
print(f"Initial version: {ds.version}")

# 2. 追加数据（创建新版本）
df2 = pd.DataFrame({
    "id": [4, 5, 6],
    "version": ["v2", "v2", "v2"],
    "type":[1, 1, 2],
})
ds = lance.write_dataset(df2, p, mode="append")
print(f"After append: {ds.version}")
ds.delete("type = 2")

✅ 成功删除目录: /tmp/day3.2.lance
Initial version: 1
After append: 2


{'num_deleted_rows': 2}

In [58]:
print_lance(p)

=== 数据集结构 ===
  day4.lance/
  _deletions/
    0-3-1734360561324442774.arrow (698 bytes)
    1-2-1536535464972903898.arrow (698 bytes)
  _versions/
    18446744073709551611.manifest (1,172 bytes)
    latest_version_hint.json (13 bytes)
    18446744073709551612.manifest (1,232 bytes)
    18446744073709551613.manifest (1,030 bytes)
    18446744073709551614.manifest (1,008 bytes)
  _transactions/
    2-c5d1c753-8c1a-4e99-a026-646d05a9158c.txn (223 bytes)
    0-ff9f525c-daf8-459e-9eb2-3cc5041beba3.txn (181 bytes)
    3-51184853-647e-4ea8-9b1d-e66aa232593a.txn (146 bytes)
    1-47890dbf-10d1-4c9d-9823-c089e34e785b.txn (122 bytes)
  data/
    011100001100000010010111065b5b4bb4843079bcdf98f375.lance (1,195 bytes)
    0101011111110101101001008fb6094e7bbb9615620f3d4aa5.lance (1,195 bytes)
    001111001100011000111100cbb3b0486586d511db5da1e9e8.lance (1,195 bytes)


In [57]:
# 2. 查看 Fragment 结构
print("\n=== Fragment 结构 ===")
for frag in ds.get_fragments():
    print(frag)
    # print(f"Fragment {frag.fragment_id}: {frag.count_rows()} rows.")
    # for file in frag.data_files():
    #     print(f"  Data file: {file.path()}")
    #     print(f"  Field IDs: {file.field_ids()}")
    #     print(f"       Details: {file}")


=== Fragment 结构 ===
LanceFileFragment(id=0, data_files=['0101011111110101101001008fb6094e7bbb9615620f3d4aa5.lance'], deletion_file='_deletions/0-3-1734360561324442774.arrow')
LanceFileFragment(id=1, data_files=['011100001100000010010111065b5b4bb4843079bcdf98f375.lance'], deletion_file='_deletions/1-2-1536535464972903898.arrow')
LanceFileFragment(id=2, data_files=['001111001100011000111100cbb3b0486586d511db5da1e9e8.lance'])


### Transaction
```
  Transaction
  ├── read_version: u64                       ← 基于哪个版本
  ├── uuid: String                            ← 唯一ID
  ├── tag: Option<String>                     ← 业务标签
  ├── transaction_properties: HashMap         ← 自定义元数据
  └── operation: Operation                    ← 操作（oneof）
      ├── Append
      │   └── fragments: [...]
      ├── Delete
      │   ├── updated_fragments: [...]
      │   ├── deleted_fragment_ids: [...]
      │   └── predicate: "id > 100"
      ├── Update
      │   ├── removed_fragment_ids: [...]
      │   ├── updated_fragments: [...]
      │   ├── new_fragments: [...]
      │   └── fields_modified: [...]
      └── ...
```

In [59]:
  ds = lance.dataset(p)
  delta = ds.delta(compared_against=0)

  for tx in delta.list_transactions():
      print(f"{'='*50}")
      print(tx)
      # print(f"Transaction UUID: {tx.uuid}")
      # print(f"Read Version: {tx.read_version}")

      # # 操作类型
      # op = tx.operation
      # print(f"Operation Type: {type(op).__name__}")
      # fragments = []
      # if hasattr(op, 'fragments') and op.fragments:
      #     fragments=op.fragments
      # elif hasattr(op, 'updated_fragments') and op.updated_fragments:
      #     fragments=op.updated_fragments
          
      # print(f"  Fragments: {len(fragments)}")
      # for frag in fragments:
      #     print(f"    Fragment {frag.id}: {frag.physical_rows} rows")
      #     for f in frag.files:
      #         print(f"      File: {f.path}")

      print()

Transaction(read_version=0, operation=LanceOperation.Overwrite(new_schema=Schema { fields: [Field { name: "id", id: 0, parent_id: -1, logical_type: LogicalType("int64"), metadata: {}, encoding: Some(Plain), nullable: true, children: [], dictionary: None, unenforced_primary_key_position: None, unenforced_clustering_key_position: None }, Field { name: "val", id: 1, parent_id: -1, logical_type: LogicalType("string"), metadata: {}, encoding: Some(VarBinary), nullable: true, children: [], dictionary: None, unenforced_primary_key_position: None, unenforced_clustering_key_position: None }], metadata: {} }, fragments=[FragmentMetadata(id=0, files=[DataFile(path='0101011111110101101001008fb6094e7bbb9615620f3d4aa5.lance', fields=[0, 1], column_indices=[0, 1], file_major_version=2, file_minor_version=1, file_size_bytes=1195)], physical_rows=3, deletion_file=None, row_id_meta=None, created_at_version_meta=None, last_updated_at_version_meta=None)], initial_bases=None), uuid='ff9f525c-daf8-459e-9eb2

rust/lance-tools/src/meta.rs#fmt
```rust
writeln!(f, "columns:")?;
for (i, col) in self.file_metadata.column_metadatas.iter().enumerate() {
    writeln!(f, "  column_metadatas {}: {} pages", i, col.pages.len())?;
    for (j, page) in col.pages.iter().enumerate() {
        writeln!(f, "    page {}: {} rows, {} buffers",
                 j, page.length, page.buffer_sizes.len())?;
        for (k, size) in page.buffer_sizes.iter().enumerate() {
            writeln!(f, "      buffer {}: {} bytes", k, size)?;
        }
    }
}

for infos in self.file_metadata.column_infos.iter() {
    writeln!(f, "  column-infos index={}, len={}", infos.index, infos.page_infos.len())?;
    for (j, info) in infos.page_infos.iter().enumerate() {
        writeln!(f, "    page-info {}: rows={}, priority={}", j, info.num_rows, info.priority)?;
        for (pos, size) in info.buffer_offsets_and_sizes.iter() {
            writeln!(f, "      position={}, size={}", pos, size)?;
        }
    }
}

writeln!(f, "Global Buffers: len={}", self.file_metadata.file_buffers.len())?;
for (i, buff) in self.file_metadata.file_buffers.iter().enumerate() {
    writeln!(f, "  buffer{}:position={}, size={}", i, buff.position, buff.size)?;
}
```

> cargo run --bin lance-tools -- file meta -s /tmp/day5_vectors.lance/data/10001011100101111011110159d2074ef7bb4bd2a686d5046a.lance

## 版本控制实践
掌握 Lance 的版本控制和时间旅行

### 时间旅行

In [60]:
import lance
import pandas as pd

p = "/tmp/day4.lance"
delete_if_exists(p)

# 1. 创建初始数据
df1 = pd.DataFrame({
    "id": [1, 2, 3],
    "val": ["v1", "v1", "v1"]
})
ds = lance.write_dataset(df1, "/tmp/day4.lance") # mode="overwrite"
print(f"Initial version: {ds.version}")

# 2. 追加数据（创建新版本）
df2 = pd.DataFrame({
    "id": [4, 5, 6],
    "val": ["v2", "v2", "v2"]
})
ds = lance.write_dataset(df2, "/tmp/day4.lance", mode="append")
print(f"After append: {ds.version}")

ds.update(dict(val = "'v2_2'"), where="id=5")

✅ 成功删除目录: /tmp/day4.lance
Initial version: 1
After append: 2


{'num_rows_updated': 1}

In [61]:
# 3. 删除数据（创建新版本）
ds.delete("id = 2")
print(f"After delete: {ds.version}")

After delete: 4


In [62]:
# 4. 查看所有版本
print("\n=== 版本历史 ===")
ds = lance.dataset("/tmp/day4.lance", version=4)
for v in ds.versions():
    print(f"  Rows: {v}")


=== 版本历史 ===
  Rows: {'version': 1, 'timestamp': datetime.datetime(2026, 5, 31, 8, 17, 23, 125207), 'metadata': {'total_data_file_rows': '3', 'total_data_files': '1', 'total_deletion_file_rows': '0', 'total_deletion_files': '0', 'total_files_size': '1195', 'total_fragments': '1', 'total_rows': '3'}}
  Rows: {'version': 2, 'timestamp': datetime.datetime(2026, 5, 31, 8, 17, 23, 131415), 'metadata': {'total_data_file_rows': '6', 'total_data_files': '2', 'total_deletion_file_rows': '0', 'total_deletion_files': '0', 'total_files_size': '2390', 'total_fragments': '2', 'total_rows': '6'}}
  Rows: {'version': 3, 'timestamp': datetime.datetime(2026, 5, 31, 8, 17, 23, 138846), 'metadata': {'total_data_file_rows': '7', 'total_data_files': '3', 'total_deletion_file_rows': '1', 'total_deletion_files': '1', 'total_files_size': '3585', 'total_fragments': '3', 'total_rows': '6'}}
  Rows: {'version': 4, 'timestamp': datetime.datetime(2026, 5, 31, 8, 17, 45, 878210), 'metadata': {'total_data_file_rows'

In [64]:
# 5. 时间旅行 - 读取旧版本
print("\n=== 时间旅行 ===")
for version in [1, 2, 3, 4]:
    old_ds = lance.dataset("/tmp/day4.lance", version=version)
    df = old_ds.to_table().to_pandas()
    print(f"Version {version}: {len(df)} rows")
    print(df)
    print()


=== 时间旅行 ===
Version 1: 3 rows
   id val
0   1  v1
1   2  v1
2   3  v1

Version 2: 6 rows
   id val
0   1  v1
1   2  v1
2   3  v1
3   4  v2
4   5  v2
5   6  v2

Version 3: 6 rows
   id   val
0   1    v1
1   2    v1
2   3    v1
3   4    v2
4   6    v2
5   5  v2_2

Version 4: 5 rows
   id   val
0   1    v1
1   3    v1
2   4    v2
3   6    v2
4   5  v2_2



- _deletions: {fragment_id}-{version}-{random_id}.arrow
- _transactions: {version}-{uuid}.txn

In [65]:
print_lance("/tmp/day4.lance")

=== 数据集结构 ===
  day4.lance/
  _deletions/
    0-3-3080038014305318860.arrow (698 bytes)
    1-2-11799146173192279608.arrow (698 bytes)
  _versions/
    18446744073709551611.manifest (1,173 bytes)
    latest_version_hint.json (13 bytes)
    18446744073709551612.manifest (1,234 bytes)
    18446744073709551613.manifest (1,030 bytes)
    18446744073709551614.manifest (1,008 bytes)
  _transactions/
    0-b377cbd0-6336-4083-ac65-8535b78a96f3.txn (181 bytes)
    1-31b9a7e2-aa5d-437a-8341-70667c5f6477.txn (122 bytes)
    3-181d1faa-e199-4617-8c95-5f42860017fe.txn (146 bytes)
    2-d840312c-7eb3-4521-baa8-2140a4c59310.txn (224 bytes)
  data/
    01101111111011111110001171168348788b2808bee5f75e05.lance (1,195 bytes)
    0011111101010011100110112446b540db88975e2b9cb0c763.lance (1,195 bytes)
    011101010100011000100011f6a5aa46dba6edc16975bbaee9.lance (1,195 bytes)


### 回滚（切回旧版本）
Lance 的版本链是线性的。你可以基于旧版本读取，但写入时 Lance 会自动把你的变更应用到最新版本，不支持在旧版本上"分叉"创建并行历史。

```
  ┌────────────────────────────┬───────────────────────────────────────┐
  │            操作             │                 代码                  │
  ├────────────────────────────┼───────────────────────────────────────┤
  │ 打开指定版本                 │ lance.dataset(uri, version=N)         │
  ├────────────────────────────┼───────────────────────────────────────┤
  │ 查看最新版本号               │ ds.version                            │
  ├────────────────────────────┼───────────────────────────────────────┤
  │ 列出所有版本                 │ ds.versions()                         │
  ├────────────────────────────┼───────────────────────────────────────┤
  │ 回滚到某版本（创建新版本）     │ ds.restore()                          │
  ├────────────────────────────┼───────────────────────────────────────┤
  │ 基于某版本写新版本            │ 不支持在旧版本上"分叉"修改。              │
  └────────────────────────────┴───────────────────────────────────────┘
```

这里会从版本2copy一份Fragments创建一个新的版本4 

In [212]:
ds_v2 = lance.dataset("/tmp/day4.lance", version=2)
ds_v2.restore() 
lance.dataset("/tmp/day4.lance").count_rows()

6

### 标签管理

In [213]:
# 创建标签管理
ds = lance.dataset("/tmp/day4.lance", version=version)
ds.tags.create("release-1.0", version=1)
print("Created tag: release-1.0")

Created tag: release-1.0


In [214]:
ds.tags.list()

{'release-1.0': {'version': 1, 'manifest_size': 841}}

In [215]:
# 通过标签读取
tagged_ds = lance.dataset("/tmp/day4.lance", version="release-1.0")
print(f"Tagged version rows: {tagged_ds.count_rows()}")

Tagged version rows: 3


In [216]:
ds = lance.dataset("/tmp/day4.lance", version=version)


### Q&A

#### Lance 的版本控制与 Git 的异同？与数据库 MVCC 的异同？


什么是MVCC？ -> 多版本并发控制（Multi-Version Concurrency Control）


Lance：基于乐观并发 + 自动 rebase 重试
- 当两个客户端基于版本 V 并发提交时，先提交者生成 V+1。
- 后提交者发现版本号已变，会：
    1. 读取对方的 transaction 文件
    2. 判断操作兼容性（Append vs Delete vs CreateIndex 等）
    3. 兼容则 rebase 到 V+1 重试，生成 V+2
    4. 不兼容（如 Overwrite）则报 CommitConflictError

```
  ┌──────────┬───────────────────────────┬──────────────────────────────┬──────────────────────────┐
  │   维度    │           Lance           │             Git              │       数据库 MVCC         │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 核心对象  │ 数据集（行+列）              │ 文件 + 目录                   │ 数据行                    │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 版本粒度  │ 整个数据集                  │ 整个仓库                      │ 单行                      │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 版本标识  │ 自增整数 version=N          │ SHA-1 哈希                   │ 事务 ID / 时间戳           │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 版本链    │ 线性                      │ 有向无环图（DAG）               │ 线性（每行独立）            │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 分支      │ ❌ 不支持分叉写             │ ✅ 任意分支                    │ ❌ 无                    │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 合并      │ ❌ 无 merge               │ ✅ merge / rebase             │ ❌ 无                    │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 存储方式  │ Manifest 快照 + 数据文件     │ 内容寻址（blob/tree/commit）  │ WAL + 元组多版本链         │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 冲突解决  │ 乐观并发 + 自动重试          │ 手动 merge 冲突               │ 锁 / 自动回滚              │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 写入模式  │ 追加新版本                  │ 追加新 commit                 │ 行级 MVCC                │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 回滚      │ restore(version=N)        │ reset --hard SHA             │ ROLLBACK                 │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 删除     │ 逻辑删除（deletion_file）    │ 仍可从历史恢复                 │ 逻辑删除（标记 deleted）    │
  ├──────────┼───────────────────────────┼──────────────────────────────┼──────────────────────────┤
  │ 垃圾回收  │ cleanup_old_versions()    │ git gc                       │ VACUUM                   │
  └──────────┴───────────────────────────┴──────────────────────────────┴──────────────────────────┘
```


| A \ B | Append | Delete | Update | Rewrite | CreateIndex | Merge | Overwrite |
|---|---|---|---|---|---|---|---|
| **Append** | ✅ | ✅ | ⚠️ 带 PK 时冲突 | ✅ | ✅ | ✅ | ❌ |
| **Delete** | ✅ | ⚠️ 重叠 fragment | ⚠️ 重叠 fragment | ⚠️ 重叠 fragment | ✅ | ❌ retryable | ❌ 不可重试 |
| **Update** | ⚠️ 带 PK 时冲突 | ⚠️ 重叠 fragment | ⚠️ 重叠 fragment 或 PK 重叠 | ⚠️ 重叠 fragment | ✅ | ❌ retryable | ❌ 不可重试 |
| **Rewrite** | ✅ | ⚠️ 重叠 fragment | ⚠️ 重叠 fragment | ⚠️ 重叠 fragment | ⚠️ | ⚠️ | ❌ |
| **CreateIndex** | ✅ | ✅ | ✅ | ⚠️ | ⚠️ 同索引冲突 | ⚠️ | ❌ |
| **Merge** | ❌ retryable | ❌ retryable | ❌ retryable | ⚠️ | ⚠️ | ❌ | ❌ |
| **Overwrite** | ❌ 不可重试 | ❌ 不可重试 | ❌ 不可重试 | ❌ | ❌ | ❌ | ❌ |

| 符号 | 含义 |
|---|---|
| ✅ | 兼容，无冲突，直接提交 |
| ⚠️ | 条件兼容（按 fragment 重叠或主键重叠判断），可能自动重试 |
| ❌ retryable | 冲突，但可重试（RetryableCommitConflict） |
| ❌ 不可重试 | 冲突，不可重试（CommitConflict），直接报错 |


计哲学差异

- Git：
   - 目标 - 内容完整性，让人决策
   - 哲学 - "我无法理解语义，请你判断"

- PostgreSQL：
   - 目标 - ACID 强一致
   - 哲学 - "我保证正确性，要么排队要么回滚"
   - Read Committed: 性能优先，自动叠加
   - Repeatable Read/Serializable: 正确性优先，宁可报错

- Lance：
   - 目标 - 数据集级原子提交
   - 哲学 - "操作大多兼容，先试试自动合并"
   - 乐观假设：并发写大多是 Append/Delete 不同 fragment
   - 最坏情况：自动重试 N 次后报错



## Index

In [ ]:
%cd /Users/juntao/src/github.com/lance/lance/

In [18]:
p="/tmp/day5_vectors.lance"
delete_if_exists(p)

import lance
import numpy as np
import pyarrow as pa

# 1. 创建带向量的数据集
n = 100_000  # 10万条
dimension = 128


# 生成向量数据
vectors = np.random.randn(n, dimension).astype(np.float32)

# 用 PyArrow 构造 FixedSizeListArray
vector_array = pa.FixedSizeListArray.from_arrays(
  vectors.flatten(),  # 展平为一维
  dimension           # 每个列表的长度
)

step = 4
integer_part = np.repeat(np.arange(0, n, step) // step * 10 + 10, step)[:n]  # 从 10 开始递增
decimal_part = np.random.uniform(0.01, 0.99, size=n) * 10


table = pa.table({
    "id": range(n),
    "text": [f"document_{i}" for i in range(n)],
    "category": np.random.choice(["tech", "science", "art", "sports"], n),
    "price": decimal_part + integer_part,
    "vector": vector_array
})

# 2. 写入 Lance（调整参数优化写入）
ds = lance.write_dataset(
    table, 
    p,
    # max_rows_per_group=8192,
    # max_rows_per_file=1024*1024
)
print(f"Schema: {ds.schema}")
print(f"Created dataset: {ds.count_rows()} rows")

✅ 成功删除目录: /tmp/day5_vectors.lance
Schema: id: int64
text: string
category: string
price: double
vector: fixed_size_list<item: float>[128]
  child 0, item: float
Created dataset: 100000 rows


In [221]:
print_lance(p)
# 3. 验证数据
print("\n=== 数据集信息 ===")
print(f"Schema: {ds.schema}")
print(f"\nSample data:")
sample = ds.to_table(limit=3).to_pandas()
print(sample)

# 4. 测试随机访问
import time
random_ids = np.random.choice(n, 100, replace=False).tolist()
start = time.time()
batch = ds.take(random_ids, columns=["id", "text", "vector"])
elapsed = time.time() - start
print(f"\nRandom access 100 rows: {elapsed*1000:.1f}ms")

=== 数据集结构 ===
  day5_vectors.lance/
  _versions/
    latest_version_hint.json (13 bytes)
    18446744073709551614.manifest (719 bytes)
  _transactions/
    0-4d13487a-6662-4780-b8c1-16f9d9cb7663.txn (317 bytes)
  data/
    1100100011110111111010108bdbbf40d1b07725f9e1010f36.lance (53,026,779 bytes)

=== 数据集信息 ===
Schema: id: int64
text: string
category: string
price: double
vector: fixed_size_list<item: float>[128]
  child 0, item: float

Sample data:
   id        text category      price  \
0   0  document_0     tech  14.995077   
1   1  document_1   sports  17.186063   
2   2  document_2   sports  11.666151   

                                              vector  
0  [1.5110006, 0.8995954, 0.12016949, -1.0522748,...  
1  [-0.26201326, -0.4548963, -0.96198505, 1.48338...  
2  [0.60827404, -1.3568507, -1.152016, 0.59609205...  

Random access 100 rows: 7.4ms


In [222]:
!cargo run --bin lance-tools -- file meta -s /tmp/day5_vectors.lance/data/0100000011011010001001115780644c6389518d7219d77254.lance

/Users/juntao/src/github.com/lance/lance/python/python/lance/__init__.py:320: UserWarning: lance is not fork-safe. If you are using multiprocessing, use spawn or forkserver instead.
  warnings.warn(


    Finished `dev` profile [unoptimized + debuginfo] target(s) in 1.79s
     Running `target/debug/lance-tools file meta -s /tmp/day5_vectors.lance/data/1100100011110111111010108bdbbf40d1b07725f9e1010f36.lance`
version: 2.1
num_rows: 100000
num_data_bytes: 53023488
num_column_metadata_bytes: 3042
num_footer_bytes: 3291
schema:
Field(id=0, name=id, type=int64)
Field(id=1, name=text, type=string)
Field(id=2, name=category, type=string)
Field(id=3, name=price, type=double)
Field(id=4, name=vector, type=fixed_size_list:float:128)
columns:
  column_metadatas 0: 1 pages
    page 0: 100000 rows, 2 buffers
      buffer 0: 196 bytes
      buffer 1: 198560 bytes
  column_metadatas 1: 1 pages
    page 0: 100000 rows, 2 buffers
      buffer 0: 740 bytes
      buffer 1: 804704 bytes
  column_metadatas 2: 1 pages
    page 0: 100000 rows, 3 buffers
      buffer 0: 196 bytes
      buffer 1: 26656 bytes
      buffer 2: 48 bytes
  column_metadatas 3: 1 pages
    page 0: 100000 rows, 2 buffers
      buff

### 创建 IVF_PQ 索引

In [5]:
print("\n=== 创建 IVF_PQ 索引 ===")
ds = lance.dataset(p)
print(f"Dataset: {ds.count_rows()} rows")
print(f"Schema: {ds.schema}")

# 检查向量列类型
print(f"Vector column type: {ds.schema.field('vector').type}")

if any(idx['name'] == 'vector_idx' for idx in ds.list_indices()):
    ds.drop_index("vector_idx")
    print("已删除索引 vector_idx")

# 创建索引
# 注意：num_sub_vectors 必须能整除 dimension (128)
# 128 / 16 = 8，所以 num_sub_vectors=16 可以
start = time.time()
ds.create_index(
  "vector",
  index_type="IVF_PQ",
  num_partitions=256,      # IVF 分区数：10万数据通常 256~512
  num_sub_vectors=16,      # PQ 子向量数：128/16=8，每子向量 8 维
  metric="l2",             # 距离度量：l2 (欧氏距离) / cosine / dot
)
print(f"索引创建耗时: {time.time() - start:.2f}s")


=== 创建 IVF_PQ 索引 ===
Dataset: 100000 rows
Schema: id: int64
text: string
category: string
price: double
vector: fixed_size_list<item: float>[128]
  child 0, item: float
Vector column type: fixed_size_list<item: float>[128]
索引创建耗时: 27.16s


In [20]:
print_lance(p)
# 查看索引信息
print("\n=== 索引信息 ===")
for idx in ds.list_indices():
    # print(idx)
    print(f"  Index name: {idx['name']}")
    print(f"  Index type: {idx['type']}")
    print(f"  Columns: {idx['fields']}")
    # 获取详细统计
    stats = ds.stats.index_stats(idx['name'])
    print(stats["num_unindexed_rows"], stats["num_indexed_rows"])

=== 数据集结构 ===
  day5_vectors.lance/
  _versions/
    latest_version_hint.json (13 bytes)
    18446744073709551613.manifest (1,207 bytes)
    18446744073709551614.manifest (718 bytes)
  _transactions/
    0-d959c924-76a3-49ef-93d9-99ab6254261d.txn (317 bytes)
    1-a40eb5cc-c85c-4721-8769-2209e923fb15.txn (421 bytes)
  data/
    0100000011011010001001115780644c6389518d7219d77254.lance (53,213,019 bytes)
  _indices/
  1a003cc9-ff5f-41a2-a025-4baed97681ce/
    auxiliary.idx (51,399,430 bytes)
    index.idx (15,602,083 bytes)

=== 索引信息 ===
  Index name: vector_idx
  Index type: IVF_HNSW_FLAT
  Columns: ['vector']
0 100000


[2026-06-13T01:45:26Z WARN  lance::index::vector::ivf] Vector index statistics currently include centroids, which can use significant memory for large indexes. In a future release, centroids will be excluded from statistics by default. Set LANCE_INCLUDE_VECTOR_CENTROIDS=true to preserve the current behavior (and silence this warning), or LANCE_INCLUDE_VECTOR_CENTROIDS=false to opt in to the new behavior now.


#### ivf-pq.md 例子
```
[index.idx]

+-----------------------------------------+
| 文件头（元数据）                           |
| - 质心: 3 个 8 维向量                     |
| - 分区数: 3                              |
| - PQ 参数: n_subvectors=2, n_bits=8      |
| - 全局 PQ 码本: 256 x 4 维 x 2 子空间      |
|   （所有分区共用）                         |
+-----------------------------------------+
| 分区 0                                   |
| offset=1024, length=~3,333               |
|                                          |
| PQ 编码 + Row ID:                         |
| [1, 0] , 0                               |
| [3, 2] , 1                               |
| [0, 2] , 4                               |
| [1, 0] , 8                               |
| [2, 1] , 11                              |
| ...                                      |
+------------------------------------------+
| 分区 1                                    |
| offset=1152, length=~3,333               |
| [0, 0] , 2                               |
| [2, 3] , 3                               |
| [0, 2] , 5                               |
| ...                                      |
+------------------------------------------+
| 分区 2                                    |
| offset=1280, length=~3,334               |
| [0, 2] , 6                               |
| [2, 1] , 7                               |
| ...                                      |
+-----------------------------------------+
```

### search

In [7]:
query = np.random.randn(dimension).astype(np.float32)

In [8]:
# 暴力扫描
result=ds.to_table(nearest={"column": "vector", "q": query, "k": 50, "nprobes": 1, "use_index": False},
  filter="category = 'art'").to_pandas()
truth=result["id"].tolist()
def recall(pred):
    pred_ids = set(truth)
    truth_ids = set(pred)
    return len(pred_ids & truth_ids) / len(truth_ids)
result

,id,text,category,price,vector,_distance
0,39188,document_39188,art,97980.225641,"[0.49567226, 0.7751246, 0.50674707, 0.2249197,...",162.703934
1,57657,document_57657,art,144158.086441,"[0.64834976, 0.005897805, 0.6382816, 1.2667183...",163.473282
2,43014,document_43014,art,107548.430469,"[1.1516612, 1.7407916, 0.14456785, -0.6721746,...",167.853302
3,14425,document_14425,art,36079.679204,"[0.64691293, 1.6747415, -0.9822388, 1.713068, ...",167.881363
4,42289,document_42289,art,105732.769794,"[0.026032297, -0.49788, -1.0384797, 0.28038105...",169.102234
5,54880,document_54880,art,137217.720289,"[0.80190855, 0.9105908, 1.1230356, 0.2872294, ...",169.679413
6,92655,document_92655,art,231647.651682,"[-1.2223647, 0.72118545, -2.1134489, -0.513534...",170.003510
7,92236,document_92236,art,230600.975596,"[-1.1799898, 0.17716229, 0.5452219, 2.0120559,...",170.030182
8,34938,document_34938,art,87353.757434,"[1.0910623, -0.83664525, -0.8675758, 1.6836572...",171.007141
9,40787,document_40787,art,101978.220650,"[0.23113164, 0.20327091, -0.2611446, -0.575739...",171.211456


In [9]:
# IVF_PQ 不传 refine
result=ds.to_table(nearest={"column": "vector", "q": [query], "k": 10, "nprobes": 1},filter="category = 'art'").to_pandas()
# 为什么 refine_factor=5 反而效果不好？ ==>  这个distance 不准确，是 PQ的模糊距离
print("recal=", recall(result["id"].tolist()))
result

recal= 0.25


,query_index,id,text,category,price,vector,_distance
0,0,69648,document_69648,art,174133.740808,"[2.2297473, 0.3752173, 0.89944005, 0.2842096, ...",161.314148
1,0,9638,document_9638,art,24101.107928,"[0.9004677, 0.8317895, -0.5787168, 0.013006139...",162.884262
2,0,39188,document_39188,art,97980.225641,"[0.49567226, 0.7751246, 0.50674707, 0.2249197,...",167.885590
3,0,76794,document_76794,art,191993.164304,"[0.5503389, 1.3536154, 1.6777217, 0.72789097, ...",173.297852


In [10]:
result=ds.to_table(nearest={"column": "vector", "q": [query], "k": 10, "nprobes": 1, "refine_factor": 10}, filter="category = 'art'").to_pandas()
print("recal=", recall(result["id"].tolist()))
result

recal= 0.25


,query_index,id,text,category,price,vector,_distance
0,0,39188,document_39188,art,97980.225641,"[0.49567226, 0.7751246, 0.50674707, 0.2249197,...",162.703934
1,0,23448,document_23448,art,58633.905489,"[1.4933126, 0.5576332, -0.94183517, -0.6909200...",188.707962
2,0,51131,document_51131,art,127831.741949,"[-0.8585484, 0.3946147, 0.72627693, 0.29965854...",194.769745
3,0,75948,document_75948,art,189888.752980,"[-0.88354, 0.0008308237, -0.92506003, 0.358067...",197.173721


In [11]:
# scanner
result = ds.scanner(nearest={"column": "vector", "q": [query], "k": 10, "nprobes": 1, "refine_factor": 10}, filter="category = 'art'"
).to_table().to_pandas()
result

,query_index,id,text,category,price,vector,_distance
0,0,39188,document_39188,art,97980.225641,"[0.49567226, 0.7751246, 0.50674707, 0.2249197,...",162.703934
1,0,23448,document_23448,art,58633.905489,"[1.4933126, 0.5576332, -0.94183517, -0.6909200...",188.707962
2,0,51131,document_51131,art,127831.741949,"[-0.8585484, 0.3946147, 0.72627693, 0.29965854...",194.769745
3,0,75948,document_75948,art,189888.752980,"[-0.88354, 0.0008308237, -0.92506003, 0.358067...",197.173721


#### ANN 和 KNN

|            | ANN (Approximate Nearest Neighbor) | KNN (K-Nearest Neighbor) |
|------------|------------------------------------|--------------------------|
| 中文       | 近似最近邻                         | 精确最近邻               |
| 算法       | 使用索引（IVF_PQ 等）              | 暴力扫描                 |
| 精度       | 近似，可能遗漏                     | 100% 精确                |
| 速度       | 快                                 | 慢                       |
| 适用数据量 | 大数据（百万级以上）               | 小数据（万级以下）       |
| 资源消耗   | 内存中缓存索引                     | 全表扫描                 |

#### KNN
flat_knn 只在有 fts_filter（全文搜索过滤）时触发:

```
# 同时有向量搜索 + 全文搜索过滤
ds.scanner(
  nearest={"column": "vector", "q": query, "k": 10},
  filter="text_column contains 'hello'",  # ← 全文搜索
  prefilter=True
)

或者：

# 使用 PhraseQuery 等全文查询
ds.scanner(
  nearest={"column": "vector", "q": query, "k": 10},
  filter=lance.FullTextQuery("hello world", "text_column"),
  prefilter=True
)
```
原因：
1. FTS 返回的文档数通常不多（高选择性）
2. 这些文档的向量需要加载（Take）
3. 文档少 → 暴力扫描比 ANN 更快（避免索引开销）


#### prefilter
If True then the filter will be applied before the vector query is run.

In [14]:
result = ds.scanner(
    nearest={"column": "vector", "q": [query], "k": 10, "nprobes": 1, "refine_factor": 10}, 
    filter="category = 'art'",
    prefilter=True
).to_table().to_pandas()
result

,query_index,id,text,category,price,vector,_distance
0,0,39188,document_39188,art,97980.225641,"[0.49567226, 0.7751246, 0.50674707, 0.2249197,...",162.703934
1,0,23448,document_23448,art,58633.905489,"[1.4933126, 0.5576332, -0.94183517, -0.6909200...",188.707962
2,0,83859,document_83859,art,209658.993520,"[0.43686965, -0.30992863, 0.88366044, -0.38934...",189.724747
3,0,51131,document_51131,art,127831.741949,"[-0.8585484, 0.3946147, 0.72627693, 0.29965854...",194.769745
4,0,75948,document_75948,art,189888.752980,"[-0.88354, 0.0008308237, -0.92506003, 0.358067...",197.173721
5,0,76794,document_76794,art,191993.164304,"[0.5503389, 1.3536154, 1.6777217, 0.72789097, ...",203.589890
6,0,12477,document_12477,art,31201.312733,"[-0.05771873, 0.9320615, 0.02438712, -0.093338...",206.948944
7,0,93326,document_93326,art,233320.638062,"[-0.40333647, 0.58657205, 1.3746555, -0.960606...",209.184067
8,0,69648,document_69648,art,174133.740808,"[2.2297473, 0.3752173, 0.89944005, 0.2842096, ...",209.713913
9,0,31462,document_31462,art,78663.336903,"[1.697577, 0.24073857, -0.20291772, -1.6039282...",210.108932


### HNSW 
（Hierarchical Navigable Small World）算法

In [19]:
start = time.time()
ds.create_index(
  "vector",
  index_type="IVF_HNSW_FLAT",
  metric="l2",             # 距离度量：l2 (欧氏距离) / cosine / dot
)
print(f"索引创建耗时: {time.time() - start:.2f}s")

索引创建耗时: 7.58s


In [23]:
print_lance(p)
!cd ../ && cargo run --bin lance-tools -- file meta -s /tmp/day5_vectors.lance/_indices/1a003cc9-ff5f-41a2-a025-4baed97681ce/auxiliary.idx

/Users/juntao/src/github.com/lance/lance/python
    Finished `dev` profile [unoptimized + debuginfo] target(s) in 0.97s
     Running `target/debug/lance-tools file meta -s /tmp/day5_vectors.lance/_indices/1a003cc9-ff5f-41a2-a025-4baed97681ce/auxiliary.idx`
version: 2.1
num_rows: 100000
num_data_bytes: 51398904
num_column_metadata_bytes: 280
num_footer_bytes: 518
schema:
Field(id=0, name=_rowid, type=uint64)
Field(id=1, name=flat, type=fixed_size_list:float:128)
columns:
  column_metadatas 0: 1 pages
    page 0: 100000 rows, 2 buffers
      buffer 0: 196 bytes
      buffer 1: 198560 bytes
  column_metadatas 1: 1 pages
    page 0: 100000 rows, 1 buffers
      buffer 0: 51200000 bytes
  column-infos index=0, len=1
    page-info 0: rows=100000, priority=0
      position=51200064, size=196
      position=51200320, size=198560
  column-infos index=1, len=1
    page-info 0: rows=100000, priority=0
      position=0, size=51200000
Global Buffers: len=2
  buffer0:position=51398912, size=166
  bu

### BTREE

In [25]:
ds.create_scalar_index("id", "BTREE")# replace=True

In [225]:
ds.to_table(filter="id=100")

pyarrow.Table
id: int64
text: string
category: string
price: double
vector: fixed_size_list<item: float>[128]
  child 0, item: float
----
id: [[100]]
text: [["document_100"]]
category: [["art"]]
price: [[262.88508065677104]]
vector: [[[0.55605143,-0.020754246,-2.0360594,-0.79008746,-1.5735472,...,-1.0292112,-0.15336084,1.0480542,0.2940029,1.6581079]]]

In [26]:
print_lance(p)

=== 数据集结构 ===
  day5_vectors.lance/
  _versions/
    latest_version_hint.json (13 bytes)
    18446744073709551612.manifest (1,122 bytes)
    18446744073709551613.manifest (1,207 bytes)
    18446744073709551614.manifest (718 bytes)
  _transactions/
    0-d959c924-76a3-49ef-93d9-99ab6254261d.txn (317 bytes)
    1-a40eb5cc-c85c-4721-8769-2209e923fb15.txn (421 bytes)
    2-c6fffe6b-7c38-460a-8659-6a87964f9f37.txn (190 bytes)
  data/
    0100000011011010001001115780644c6389518d7219d77254.lance (53,213,019 bytes)
  _indices/
  1a003cc9-ff5f-41a2-a025-4baed97681ce/
    auxiliary.idx (51,399,430 bytes)
    index.idx (15,602,083 bytes)
  f21498a5-71a6-4e3c-88a8-534ab7d3e8e5/
    page_lookup.lance (1,709 bytes)
    page_data.lance (398,089 bytes)


In [27]:
!cd ../ && cargo run --bin lance-tools -- file meta -s /tmp/day5_vectors.lance/data/0100000011011010001001115780644c6389518d7219d77254.lance

    Finished `dev` profile [unoptimized + debuginfo] target(s) in 1.06s
     Running `target/debug/lance-tools file meta -s /tmp/day5_vectors.lance/data/0100000011011010001001115780644c6389518d7219d77254.lance`
version: 2.1
num_rows: 100000
num_data_bytes: 53209728
num_column_metadata_bytes: 3042
num_footer_bytes: 3291
schema:
Field(id=0, name=id, type=int64)
Field(id=1, name=text, type=string)
Field(id=2, name=category, type=string)
Field(id=3, name=price, type=double)
Field(id=4, name=vector, type=fixed_size_list:float:128)
columns:
  column_metadatas 0: 1 pages
    page 0: 100000 rows, 2 buffers
      buffer 0: 196 bytes
      buffer 1: 198560 bytes
  column_metadatas 1: 1 pages
    page 0: 100000 rows, 2 buffers
      buffer 0: 782 bytes
      buffer 1: 990904 bytes
  column_metadatas 2: 1 pages
    page 0: 100000 rows, 3 buffers
      buffer 0: 196 bytes
      buffer 1: 26656 bytes
      buffer 2: 48 bytes
  column_metadatas 3: 1 pages
    page 0: 100000 rows, 2 buffers
      buff

In [28]:
ds.list_indices()

[{'name': 'vector_idx',
  'type': 'IVF_HNSW_FLAT',
  'uuid': '1a003cc9-ff5f-41a2-a025-4baed97681ce',
  'fields': ['vector'],
  'version': 1,
  'fragment_ids': {0},
  'base_id': None},
 {'name': 'id_idx',
  'type': 'BTree',
  'uuid': 'f21498a5-71a6-4e3c-88a8-534ab7d3e8e5',
  'fields': ['id'],
  'version': 2,
  'fragment_ids': {0},
  'base_id': None}]

In [29]:
!cd ../ && cargo run --bin lance-tools -- file meta -s /tmp/day5_vectors.lance/_indices/f21498a5-71a6-4e3c-88a8-534ab7d3e8e5/page_lookup.lance

    Finished `dev` profile [unoptimized + debuginfo] target(s) in 1.10s
     Running `target/debug/lance-tools file meta -s /tmp/day5_vectors.lance/_indices/f21498a5-71a6-4e3c-88a8-534ab7d3e8e5/page_lookup.lance`
version: 2.1
num_rows: 25
num_data_bytes: 960
num_column_metadata_bytes: 508
num_footer_bytes: 749
schema:
Field(id=0, name=min, type=int64)
Field(id=1, name=max, type=int64)
Field(id=2, name=null_count, type=uint32)
Field(id=3, name=page_idx, type=uint32)
columns:
  column_metadatas 0: 1 pages
    page 0: 25 rows, 2 buffers
      buffer 0: 2 bytes
      buffer 1: 208 bytes
  column_metadatas 1: 1 pages
    page 0: 25 rows, 2 buffers
      buffer 0: 2 bytes
      buffer 1: 208 bytes
  column_metadatas 2: 1 pages
    page 0: 25 rows, 2 buffers
      buffer 0: 2 bytes
      buffer 1: 16 bytes
  column_metadatas 3: 1 pages
    page 0: 25 rows, 2 buffers
      buffer 0: 2 bytes
      buffer 1: 112 bytes
  column-infos index=0, len=1
    page-info 0: rows=25, priority=0
      posit

In [30]:
!cd ../ && cargo run --bin lance-tools -- file meta -s /tmp/day5_vectors.lance/_indices/f21498a5-71a6-4e3c-88a8-534ab7d3e8e5/page_data.lance

    Finished `dev` profile [unoptimized + debuginfo] target(s) in 1.13s
     Running `target/debug/lance-tools file meta -s /tmp/day5_vectors.lance/_indices/f21498a5-71a6-4e3c-88a8-534ab7d3e8e5/page_data.lance`
version: 2.1
num_rows: 100000
num_data_bytes: 397696
num_column_metadata_bytes: 269
num_footer_bytes: 393
schema:
Field(id=0, name=values, type=int64)
Field(id=1, name=ids, type=uint64)
columns:
  column_metadatas 0: 1 pages
    page 0: 100000 rows, 2 buffers
      buffer 0: 196 bytes
      buffer 1: 198560 bytes
  column_metadatas 1: 1 pages
    page 0: 100000 rows, 2 buffers
      buffer 0: 196 bytes
      buffer 1: 198560 bytes
  column-infos index=0, len=1
    page-info 0: rows=100000, priority=0
      position=0, size=196
      position=256, size=198560
  column-infos index=1, len=1
    page-info 0: rows=100000, priority=0
      position=198848, size=196
      position=199104, size=198560
Global Buffers: len=1
  buffer0:position=397696, size=68


### ZONEMAP 索引（存储 min/max）

In [234]:
# ZONEMAP 存储每个 zone 的统计信息：
# - min, max
# - null_count, nan_count
# - fragment_id, local_row_offset

ds.create_scalar_index("price", "ZONEMAP")
# ds.create_scalar_index("price", "BTREE")

In [233]:
!cd ../ && cargo run --bin lance-tools -- file meta -s /tmp/day5_vectors.lance/_indices/cb7841da-8fb4-4083-b56e-5934a02e88ef/zonemap.lance

    Finished `dev` profile [unoptimized + debuginfo] target(s) in 1.45s
     Running `target/debug/lance-tools file meta -s /tmp/day5_vectors.lance/_indices/cb7841da-8fb4-4083-b56e-5934a02e88ef/zonemap.lance`
version: 2.1
num_rows: 13
num_data_bytes: 1088
num_column_metadata_bytes: 897
num_footer_bytes: 1229
schema:
Field(id=0, name=min, type=double)
Field(id=1, name=max, type=double)
Field(id=2, name=null_count, type=uint32)
Field(id=3, name=nan_count, type=uint32)
Field(id=4, name=fragment_id, type=uint64)
Field(id=5, name=zone_start, type=uint64)
Field(id=6, name=zone_length, type=uint64)
columns:
  column_metadatas 0: 1 pages
    page 0: 13 rows, 2 buffers
      buffer 0: 2 bytes
      buffer 1: 112 bytes
  column_metadatas 1: 1 pages
    page 0: 13 rows, 2 buffers
      buffer 0: 2 bytes
      buffer 1: 112 bytes
  column_metadatas 2: 1 pages
    page 0: 13 rows, 2 buffers
      buffer 0: 2 bytes
      buffer 1: 16 bytes
  column_metadatas 3: 1 pages
    page 0: 13 rows, 2 buffers

In [240]:
# 查询时会用 min/max 过滤不需要的 zone
ds.to_table(filter="price > 100 AND price < 104")

pyarrow.Table
id: int64
text: string
category: string
price: double
vector: fixed_size_list<item: float>[128]
  child 0, item: float
----
id: [[36]]
text: [["document_36"]]
category: [["art"]]
price: [[101.83817100217237]]
vector: [[[-0.5715964,1.6450565,0.09048141,0.3566613,-1.6395203,...,0.5195286,-1.4460006,0.09768119,0.14994887,-2.407026]]]

### 全文搜索

In [32]:
p="/tmp/day12_docs.lance"
delete_if_exists(p)

import lance
import pandas as pd

# 创建带文本的数据
docs = pd.DataFrame({
    "id": range(1000),
    "title": [
        "Machine Learning Basics",
        "Deep Learning Tutorial",
        "Introduction to Python",
        "Rust Programming Guide",
        "Paimon Concurrency Patterns",
    ] * 200,
    "content": [
        "This article covers the fundamentals of machine learning algorithms.",
        "A comprehensive guide to neural networks and deep learning.",
        "Python is a versatile programming language for data science.",
        "Rust provides memory safety without garbage collection.",
        "Paimon offers robust concurrency primitives for multi-threading.",
    ] * 200
})

# 取前1000条
docs = docs.head(1000)
ds = lance.write_dataset(docs, p, mode="overwrite")

✅ 成功删除目录: /tmp/day12_docs.lance


[2026-06-13T02:15:49Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/day12_docs.lance, it will be created


In [35]:
# 创建全文索引
ds.create_scalar_index("title", index_type="INVERTED")
ds.create_scalar_index("content", index_type="INVERTED")
print("Full-text indexes created")

# 全文搜索
results = ds.scanner(
    filter="title LIKE '%Patterns%'"
).to_table()
print(f"\nDocuments with 'Learning' in title: {len(results)}")
print(results.to_pandas().head())

Full-text indexes created

Documents with 'Learning' in title: 200
   id                        title  \
0   4  Paimon Concurrency Patterns   
1   9  Paimon Concurrency Patterns   
2  14  Paimon Concurrency Patterns   
3  19  Paimon Concurrency Patterns   
4  24  Paimon Concurrency Patterns   

                                             content  
0  Paimon offers robust concurrency primitives fo...  
1  Paimon offers robust concurrency primitives fo...  
2  Paimon offers robust concurrency primitives fo...  
3  Paimon offers robust concurrency primitives fo...  
4  Paimon offers robust concurrency primitives fo...  


## 编码策略

In [115]:
import lance
import pyarrow as pa
import numpy as np
import os

# 1. 创建测试数据
n = 1_000_000
data = {
    "id": range(n),
    "constant_val": [42] * n,  # 常量值
    "random_float": np.random.randn(n).astype(np.float32),
    "category": np.random.choice(["A", "B", "C", "D"], n),
    "text": [f"text_{i}" for i in range(n)],
}

# 2. 默认编码写入
ds_default = lance.write_dataset(
    pd.DataFrame(data), 
    "/tmp/day6_default.lance",
    mode="overwrite"
)

# 3. 自定义编码 - 压缩字符串
schema_compressed = pa.schema([
    pa.field("id", pa.int64()),
    pa.field("constant_val", pa.int64()),
    pa.field("random_float", pa.float32(), metadata={
        "lance-encoding:bss": "auto"  # Byte Stream Split for floats
    }),
    pa.field("category", pa.string(), metadata={
        "lance-encoding:compression": "fsst"  # FSST for strings
    }),
    pa.field("text", pa.string(), metadata={
        "lance-encoding:compression": "zstd",
        "lance-encoding:compression-level": "3"
    }),
])

ds_compressed = lance.write_dataset(
    pa.Table.from_pydict(data, schema=schema_compressed),
    "/tmp/day6_compressed.lance",
    mode="overwrite"
)

# 4. 对比文件大小
print("=== 文件大小对比 ===")
for name, path in [("default", "/tmp/day6_default.lance"), 
                    ("compressed", "/tmp/day6_compressed.lance")]:
    total_size = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total_size += os.path.getsize(os.path.join(root, f))
    print(f"{name}: {total_size / 1024 / 1024:.1f} MB")

# 5. 读取性能对比
import time

ds_d = lance.dataset("/tmp/day6_default.lance")
ds_c = lance.dataset("/tmp/day6_compressed.lance")

# 随机访问对比
random_ids = np.random.choice(n, 1000, replace=False).tolist()

start = time.time()
ds_d.take(random_ids, columns=["text"])
t_default = time.time() - start

start = time.time()
ds_c.take(random_ids, columns=["text"])
t_compressed = time.time() - start

print(f"\n=== 读取性能 ===")
print(f"Default: {t_default*1000:.1f}ms")
print(f"Compressed: {t_compressed*1000:.1f}ms")

=== 文件大小对比 ===
default: 55.9 MB
compressed: 44.4 MB

=== 读取性能 ===
Default: 14.4ms
Compressed: 13.9ms


## PyTorch

In [30]:
p="/tmp/day5_vectors.lance"
ds = lance.dataset(p)
batches = list(ds.scanner(columns=["id", "category","vector"],batch_size=1024).to_batches())
print("len", len(batches))
print(batches[0].to_pandas())
print("="*50 + "\n")
print(batches[-1].to_pandas())
print(torch.from_numpy(batches[0]['vector'].values.to_numpy()).reshape(-1,128).shape)
batches[0]['id'].to_numpy()

len 98
        id category                                             vector
0        0   sports  [0.8224635, -0.16247645, -0.7770147, -1.64866,...
1        1      art  [-0.18729043, -0.50987005, 0.6767861, 1.783381...
2        2  science  [0.65186673, 0.48001218, 1.0083715, 0.07323849...
3        3      art  [-0.7423125, -0.1472216, -1.1398886, 0.2798446...
4        4      art  [-0.70997375, -0.107787594, 0.90687805, 1.0903...
...    ...      ...                                                ...
1019  1019   sports  [-2.2096906, -1.0601747, 0.5570243, 0.28166583...
1020  1020     tech  [1.6732275, -0.34833464, 1.347759, 0.040454593...
1021  1021   sports  [-0.20065619, 0.6837119, 0.6548784, -1.0749406...
1022  1022   sports  [-0.4073255, 0.8397634, 0.7961224, -0.2935979,...
1023  1023  science  [-0.44337612, -0.3552541, -1.6620282, 1.247704...

[1024 rows x 3 columns]

        id category                                             vector
0    99328   sports  [-1.3963542, -0.3200436

array([   0,    1,    2, ..., 1021, 1022, 1023])

In [39]:
import lance
import numpy as np
from torch.utils.data import IterableDataset, DataLoader
import torch
import torch.nn as nn

# 1. 创建 Lance Dataset 包装器
class LanceIterableDataset(IterableDataset):
    def __init__(self, uri, columns, batch_size=256):
        self.ds = lance.dataset(uri)
        self.columns = columns
        self.batch_size = batch_size
  
    def __iter__(self):
        for batch in self.ds.scanner(
            columns=self.columns,
            batch_size=self.batch_size
        ).to_batches():
            yield torch.from_numpy(batch['vector'].values.to_numpy()).reshape(-1, 128)

# 2. 使用 DataLoader
dataset = LanceIterableDataset(p, columns=["id", "category", "vector"])
loader = DataLoader(dataset, batch_size=None)

# 3. 模拟训练循环
print("=== 模拟训练循环 ===")
for i, batch in enumerate(loader):
    if i >= 3:  # 只跑3个batch
        break
    print(f"Batch {i}: vector shape={batch.shape}")

=== 模拟训练循环 ===
Batch 0: vector shape=torch.Size([256, 128])
Batch 1: vector shape=torch.Size([256, 128])
Batch 2: vector shape=torch.Size([256, 128])


In [38]:
# 4. 向量数据加载（用于相似度学习）
class VectorLanceDataset(IterableDataset):
    def __init__(self, uri, batch_size=256):
        self.ds = lance.dataset(uri)
        self.batch_size = batch_size
  
    def __iter__(self):
        for batch in self.ds.scanner(
            columns=["vector", "category"],
            batch_size=self.batch_size
        ).to_batches():
            vectors = torch.tensor(
                np.vstack(batch["vector"].to_pylist()),
                dtype=torch.float32
            )
            # 将类别转为数字
            categories = batch["category"].to_pylist()
            category_map = {"tech": 0, "science": 1, "art": 2, "sports": 3}
            labels = torch.tensor(
                [category_map.get(c, 0) for c in categories],
                dtype=torch.long
            )
            yield vectors, labels

# 测试向量加载
vec_dataset = VectorLanceDataset(p)
vec_loader = DataLoader(vec_dataset, batch_size=None)

print("\n=== 向量数据加载 ===")
for i, (vectors, labels) in enumerate(vec_loader):
    if i >= 2:
        break
    print(f"Batch {i}: vectors shape={vectors.shape}, "
          f"labels shape={labels.shape}")


=== 向量数据加载 ===
Batch 0: vectors shape=torch.Size([256, 128]), labels shape=torch.Size([256])
Batch 1: vectors shape=torch.Size([256, 128]), labels shape=torch.Size([256])


## TODO